# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Remove the headers from each

In [6]:
citationHeader = rddCitations.first()
patentHeader = rddPatents.first()

print(citationHeader)
print(patentHeader)

"CITING","CITED"
"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"


In [7]:
rddCitations = rddCitations.filter(
    lambda x: x != citationHeader
)

rddPatents = rddPatents.filter(
    lambda x: x != patentHeader
)

In [8]:
rddCitations.take(2)

['3858241,956203', '3858241,1324234']

In [9]:
rddPatents.take(2)

['3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,']

## Split each line to have columns instead of , and one long string

In [10]:
rddCitations = rddCitations.map(
    lambda x: x.split(",")
)

rddPatents = rddPatents.map(
    lambda x: x.split(",")
)

In [11]:
rddCitations.take(2)

[['3858241', '956203'], ['3858241', '1324234']]

In [12]:
rddPatents.take(2)

[['3070801',
  '1963',
  '1096',
  '',
  '"BE"',
  '""',
  '',
  '1',
  '',
  '269',
  '6',
  '69',
  '',
  '1',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 ['3070802',
  '1963',
  '1096',
  '',
  '"US"',
  '"TX"',
  '',
  '1',
  '',
  '2',
  '6',
  '63',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '']]

## Patent state RDD

In [13]:
patentStates = rddPatents.map(
    lambda x: (
        x[0],
        x[5].strip().strip('"')
    )
).cache()

In [14]:
patentStates.take(5)

[('3070801', ''),
 ('3070802', 'TX'),
 ('3070803', 'IL'),
 ('3070804', 'OH'),
 ('3070805', 'CA')]

## Citation state RDD

In [15]:
citationPairs = rddCitations.map(
    lambda x: (
        x[1],
        x[0]
    )
).cache()
citationPairs.take(5)

[('956203', '3858241'),
 ('1324234', '3858241'),
 ('3398406', '3858241'),
 ('3557384', '3858241'),
 ('3634889', '3858241')]

## Join citations with parent for cited

In [16]:
citedJoin = citationPairs.join(patentStates)
citedJoin.take(4)

[('3706104', ('3858248', 'OR')),
 ('3706104', ('4681587', 'OR')),
 ('3706104', ('4795464', 'OR')),
 ('3706104', ('5370688', 'OR'))]

## Get (citing number, citing state) to join with the citing partent

In [17]:
citingWithCitedState = citedJoin.map(
    lambda x: (
        x[1][0],
        x[1][1]
    )
)
citingWithCitedState.take(4)

[('3858275', 'CA'), ('3870973', 'CA'), ('3861121', 'MO'), ('3916598', 'MO')]

In [18]:
sameState = citingWithCitedState.join(
    patentStates
)
sameState.take(5)

[('5814889', ('NY', 'FL')),
 ('5814889', ('NY', 'FL')),
 ('5814889', ('IN', 'FL')),
 ('5814889', ('NY', 'FL')),
 ('5814889', ('NY', 'FL'))]

## Keep when states are the same

In [19]:
sameState = sameState.filter(
    lambda x:
        x[1][0] != "" and
        x[1][1] != "" and
        x[1][0] == x[1][1]
)
sameState.take(5)

[('5408329', ('NY', 'NY')),
 ('5408329', ('NY', 'NY')),
 ('5408329', ('NY', 'NY')),
 ('5408329', ('NY', 'NY')),
 ('5408329', ('NY', 'NY'))]

## In order to count the citations I will map out the citing number to a 1 for every occurance that citing and cited states are the same

In [20]:
sameState = sameState.map(
    lambda x: (
        x[0],
        1
    )
)
sameStateCount = sameState.reduceByKey(
    lambda a, b: a + b
).cache()
sameStateCount.take(10)

[('5424223', 3),
 ('5968619', 6),
 ('4091802', 1),
 ('4887753', 1),
 ('5354557', 8),
 ('5792002', 1),
 ('5705130', 2),
 ('5876905', 15),
 ('5644922', 1),
 ('4022638', 1)]

## Join the counts back to the patents 

In [21]:
patentRecords = rddPatents.map(
    lambda x: (
        x[0],
        x
    )
)
result = patentRecords.leftOuterJoin(
    sameStateCount
)
result.take(3)

[('3083011',
  (['3083011',
    '1963',
    '1180',
    '',
    '"US"',
    '"PA"',
    '',
    '2',
    '',
    '271',
    '5',
    '51',
    '',
    '4',
    '',
    '0.375',
    '',
    '',
    '',
    '',
    '',
    '',
    ''],
   None)),
 ('3085036',
  (['3085036',
    '1963',
    '1194',
    '',
    '"IT"',
    '""',
    '',
    '3',
    '',
    '148',
    '5',
    '52',
    '',
    '1',
    '',
    '0',
    '',
    '',
    '',
    '',
    '',
    '',
    ''],
   None)),
 ('3087108',
  (['3087108',
    '1963',
    '1208',
    '',
    '"US"',
    '"MD"',
    '',
    '1',
    '',
    '323',
    '4',
    '45',
    '',
    '8',
    '',
    '0.6875',
    '',
    '',
    '',
    '',
    '',
    '',
    ''],
   None))]

## Ensure the count is at the end of the patent table

In [22]:
new_patents = result.map(
    lambda x: (
        x[0],
        x[1][0] + [
            x[1][1] if x[1][1] is not None else 0
        ]
    )
)
for line in new_patents.take(5):
    print(line)

('3071687', ['3071687', '1963', '1096', '', '"US"', '"OK"', '', '2', '', '376', '4', '44', '', '0', '', '', '', '', '', '', '', '', '', 0])
('3073983', ['3073983', '1963', '1110', '', '"US"', '"CA"', '', '1', '', '313', '4', '42', '', '0', '', '', '', '', '', '', '', '', '', 0])
('3075210', ['3075210', '1963', '1124', '', '"US"', '"MA"', '', '2', '', '12', '6', '63', '', '0', '', '', '', '', '', '', '', '', '', 0])
('3076765', ['3076765', '1963', '1131', '', '"US"', '"NJ"', '', '2', '', '252', '1', '19', '', '1', '', '0', '', '', '', '', '', '', '', 0])
('3076926', ['3076926', '1963', '1131', '', '"GB"', '""', '', '3', '', '323', '4', '45', '', '0', '', '', '', '', '', '', '', '', '', 0])


## Sort by the count in descending order

In [24]:
top10 = new_patents.sortBy(
    lambda x: x[1][-1],
    ascending=False
).take(10)

for line in top10:
    print(line)

('5959466', ['5959466', '1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125])
('5983822', ['5983822', '1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103])
('6008204', ['6008204', '1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100])
('5952345', ['5952345', '1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98])
('5958954', ['5958954', '1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96])
('5998655', ['5998655', '1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5